# MODULE 3 : Développement Applicatif
## V : POO avancée et finalisation de l'application

**Rappel du contexte :** Vous avez une interface fonctionnelle connectée à une logique métier solide. Aujourd'hui, on **enrichit** cette logique avec des fonctionnalités que tout vrai logiciel de gestion possède : recherche, filtre, modification, suppression, export, et on **finalise** l'application avant d'y brancher une vraie base de données demain.

### Objectifs du jour
- Ajouter des méthodes de recherche/filtre à la logique métier
- Gérer la modification et la suppression d'objets.
- Exporter des données (CSV).
- Finaliser l'interface Tkinter avec ces nouvelles fonctionnalités

### Déroulé de la journée
1. Recherche et filtrage d'objets 
2. Modification et suppression avec règles métier 
3. Export de données en CSV 
4. Atelier : finalisation de l'interface complète
5. Exercice

---
## 1. Recherche et filtrage d'objets

Une liste de 5 clients se parcourt à l'œil. Une liste de 5000 clients, non. Toute application de gestion sérieuse propose une recherche.

In [1]:
import re

def valider_email(email):
    motif = r"^[\w.\-]+@[\w\-]+\.[a-zA-Z]{2,}$"
    return re.match(motif, email) is not None

def valider_telephone(telephone):
    return telephone.isdigit() and len(telephone) == 10


class Client:
    def __init__(self, id_client, nom, email, telephone):
        if not nom or not nom.strip():
            raise ValueError("Le nom du client est obligatoire.")
        if not valider_email(email):
            raise ValueError(f"Email invalide : {email}")
        if not valider_telephone(telephone):
            raise ValueError(f"Téléphone invalide : {telephone}")
        self.id_client = id_client
        self.nom = nom.strip()
        self.email = email
        self.telephone = telephone
        self.commandes = []

    def ajouter_commande(self, commande):
        self.commandes.append(commande)

    def total_depense(self):
        return round(sum(c.total() for c in self.commandes), 2)

    def __repr__(self):
        return f"Client(id={self.id_client}, nom='{self.nom}')"


class Commande:
    def __init__(self, id_commande, client, date, produit, quantite, prix_unitaire):
        if quantite <= 0:
            raise ValueError("La quantité doit être strictement positive.")
        if prix_unitaire < 0:
            raise ValueError("Le prix ne peut pas être négatif.")
        self.id_commande = id_commande
        self.client = client
        self.date = date
        self.produit = produit
        self.quantite = quantite
        self.prix_unitaire = prix_unitaire

    def total(self):
        return round(self.quantite * self.prix_unitaire, 2)

    def __repr__(self):
        return f"Commande(id={self.id_commande}, produit='{self.produit}')"



In [2]:
class GestionCommerciale:
    def __init__(self):
        self._clients = {}
        self._prochain_id_client = 1
        self._prochain_id_commande = 1

    def ajouter_client(self, nom, email, telephone):
        for client in self._clients.values():
            if client.email == email:
                raise ValueError(f"Un client avec l'email {email} existe déjà.")
        client = Client(self._prochain_id_client, nom, email, telephone)
        self._clients[client.id_client] = client
        self._prochain_id_client += 1
        return client

    def ajouter_commande(self, id_client, date, produit, quantite, prix_unitaire):
        if id_client not in self._clients:
            raise ValueError(f"Client inconnu (id={id_client}).")
        client = self._clients[id_client]
        commande = Commande(self._prochain_id_commande, client, date, produit, quantite, prix_unitaire)
        client.ajouter_commande(commande)
        self._prochain_id_commande += 1
        return commande

    def lister_clients(self):
        return list(self._clients.values())

    def obtenir_client(self, id_client):
        return self._clients.get(id_client)



    def rechercher_clients(self, terme):
        """
        Recherche des clients par nom ou email (insensible à la casse, recherche partielle).
        Retourne la liste des clients correspondants.
        """
        terme = terme.lower().strip()
        if not terme:
            return self.lister_clients()
        return [
            c for c in self._clients.values()
            if terme in c.nom.lower() or terme in c.email.lower()
        ]

    def clients_sans_commande(self):
        """Retourne les clients qui n'ont jamais commandé (utile pour des relances commerciales)."""
        return [c for c in self._clients.values() if not c.commandes]

    def clients_par_depense_decroissante(self):
        """Retourne les clients triés du plus gros au plus petit acheteur."""
        return sorted(self._clients.values(), key=lambda c: c.total_depense(), reverse=True)

    def meilleur_client(self):
        if not self._clients:
            return None
        return max(self._clients.values(), key=lambda c: c.total_depense())


# Démonstration
gestion = GestionCommerciale()
c1 = gestion.ajouter_client("Amina Traoré", "amina@example.com", "0612345678")
c2 = gestion.ajouter_client("Jean Kouassi", "jean.k@example.com", "0623456789")
c3 = gestion.ajouter_client("Amadou Diallo", "amadou.d@example.com", "0634567890")

gestion.ajouter_commande(c1.id_client, "2026-08-20", "Clavier mécanique", 2, 45.0)
gestion.ajouter_commande(c2.id_client, "2026-08-21", "Écran 24 pouces", 1, 150.0)

print("Recherche 'ama' :", gestion.rechercher_clients("ama"))
print("\nClients sans commande :", gestion.clients_sans_commande())
print("\nClassement par dépense :", gestion.clients_par_depense_decroissante())

Recherche 'ama' : [Client(id=3, nom='Amadou Diallo')]

Clients sans commande : [Client(id=3, nom='Amadou Diallo')]

Classement par dépense : [Client(id=2, nom='Jean Kouassi'), Client(id=1, nom='Amina Traoré'), Client(id=3, nom='Amadou Diallo')]


---
## 2. Modification et suppression avec règles métier

En entreprise, on ne supprime **jamais** brutalement des données sans réfléchir aux conséquences. Exemple de règle métier réaliste : *on ne supprime pas un client qui a des commandes, car cela romprait l'historique comptable*.

In [3]:
class GestionCommercialeV2(GestionCommerciale):
    """Extension avec modification/suppression sécurisées."""

    def modifier_client(self, id_client, nom=None, email=None, telephone=None):
        """
        Modifie les champs fournis d'un client existant.
        Seuls les champs non None sont modifiés (mise à jour partielle).
        """
        client = self._clients.get(id_client)
        if client is None:
            raise ValueError(f"Client inconnu (id={id_client}).")

        if email is not None and email != client.email:
            if not valider_email(email):
                raise ValueError(f"Email invalide : {email}")
            for autre in self._clients.values():
                if autre.id_client != id_client and autre.email == email:
                    raise ValueError(f"Un autre client utilise déjà l'email {email}.")
            client.email = email

        if telephone is not None:
            if not valider_telephone(telephone):
                raise ValueError(f"Téléphone invalide : {telephone}")
            client.telephone = telephone

        if nom is not None:
            if not nom.strip():
                raise ValueError("Le nom ne peut pas être vide.")
            client.nom = nom.strip()

        return client

    def supprimer_client(self, id_client):
        """
        Supprime un client, SAUF s'il a des commandes (règle métier : préserver l'historique).
        """
        client = self._clients.get(id_client)
        if client is None:
            raise ValueError(f"Client inconnu (id={id_client}).")
        if client.commandes:
            raise ValueError(
                f"Impossible de supprimer '{client.nom}' : il a {len(client.commandes)} "
                "commande(s) enregistrée(s). Envisagez de l'archiver plutôt."
            )
        del self._clients[id_client]


# Démonstration
gestion2 = GestionCommercialeV2()
cA = gestion2.ajouter_client("Fatou Ndiaye", "fatou@example.com", "0645678901")
cB = gestion2.ajouter_client("Client Test", "test@example.com", "0656789012")
gestion2.ajouter_commande(cA.id_client, "2026-08-22", "Casque audio", 1, 60.0)

gestion2.modifier_client(cA.id_client, telephone="0699999999")
print("Client modifié :", gestion2.obtenir_client(cA.id_client))

try:
    gestion2.supprimer_client(cA.id_client)
except ValueError as erreur:
    print(f"\nSuppression bloquée comme attendu : {erreur}")

gestion2.supprimer_client(cB.id_client)  # celui-ci n'a pas de commande -> OK
print("\nClients restants :", gestion2.lister_clients())

Client modifié : Client(id=1, nom='Fatou Ndiaye')

Suppression bloquée comme attendu : Impossible de supprimer 'Fatou Ndiaye' : il a 1 commande(s) enregistrée(s). Envisagez de l'archiver plutôt.

Clients restants : [Client(id=1, nom='Fatou Ndiaye')]


---
## 3. Export de données en CSV

Le service commercial voudra sûrement exporter la liste de ses clients vers Excel. Le format CSV est le standard universel pour ça.

In [4]:
import csv

def exporter_clients_csv(gestion, chemin_fichier):
    """
    Exporte la liste des clients au format CSV, ouvrable dans Excel.

    Args:
        gestion (GestionCommerciale): l'instance contenant les clients
        chemin_fichier (str): chemin du fichier CSV à créer
    """
    with open(chemin_fichier, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Nom", "Email", "Téléphone", "Total dépensé (€)"])  # en-tête
        for client in gestion.lister_clients():
            writer.writerow([client.id_client, client.nom, client.email,
                              client.telephone, client.total_depense()])


exporter_clients_csv(gestion, "export_clients_demo.csv")

# Vérification : on relit le fichier
with open("export_clients_demo.csv", encoding="utf-8") as f:
    print(f.read())

ID,Nom,Email,Téléphone,Total dépensé (€)
1,Amina Traoré,amina@example.com,0612345678,90.0
2,Jean Kouassi,jean.k@example.com,0623456789,150.0
3,Amadou Diallo,amadou.d@example.com,0634567890,0



---
## 4.  Interface complète de l'

On regénère `gestion_commerciale.py` avec toutes les nouvelles méthodes, puis on met à jour l'interface avec une **barre de recherche** et un **bouton d'export**.

In [3]:
%%writefile gestion_commerciale.py
"""
Module métier complet : logique de gestion des clients et commandes.
Version finale de la semaine 1 (recherche, modification, suppression, export).
"""
import re
import csv


def valider_email(email):
    motif = r"^[\w.\-]+@[\w\-]+\.[a-zA-Z]{2,}$"
    return re.match(motif, email) is not None


def valider_telephone(telephone):
    return telephone.isdigit() and len(telephone) == 10


class Client:
    def __init__(self, id_client, nom, email, telephone):
        if not nom or not nom.strip():
            raise ValueError("Le nom du client est obligatoire.")
        if not valider_email(email):
            raise ValueError(f"Email invalide : {email}")
        if not valider_telephone(telephone):
            raise ValueError(f"Téléphone invalide : {telephone}")
        self.id_client = id_client
        self.nom = nom.strip()
        self.email = email
        self.telephone = telephone
        self.commandes = []

    def ajouter_commande(self, commande):
        self.commandes.append(commande)

    def total_depense(self):
        return round(sum(c.total() for c in self.commandes), 2)

    def __repr__(self):
        return f"Client(id={self.id_client}, nom='{self.nom}')"


class Commande:
    def __init__(self, id_commande, client, date, produit, quantite, prix_unitaire):
        if quantite <= 0:
            raise ValueError("La quantité doit être strictement positive.")
        if prix_unitaire < 0:
            raise ValueError("Le prix ne peut pas être négatif.")
        self.id_commande = id_commande
        self.client = client
        self.date = date
        self.produit = produit
        self.quantite = quantite
        self.prix_unitaire = prix_unitaire

    def total(self):
        return round(self.quantite * self.prix_unitaire, 2)

    def __repr__(self):
        return f"Commande(id={self.id_commande}, produit='{self.produit}')"


class GestionCommerciale:
    def __init__(self):
        self._clients = {}
        self._prochain_id_client = 1
        self._prochain_id_commande = 1

    def ajouter_client(self, nom, email, telephone):
        for client in self._clients.values():
            if client.email == email:
                raise ValueError(f"Un client avec l'email {email} existe déjà.")
        client = Client(self._prochain_id_client, nom, email, telephone)
        self._clients[client.id_client] = client
        self._prochain_id_client += 1
        return client

    def ajouter_commande(self, id_client, date, produit, quantite, prix_unitaire):
        if id_client not in self._clients:
            raise ValueError(f"Client inconnu (id={id_client}).")
        client = self._clients[id_client]
        commande = Commande(self._prochain_id_commande, client, date, produit, quantite, prix_unitaire)
        client.ajouter_commande(commande)
        self._prochain_id_commande += 1
        return commande

    def lister_clients(self):
        return list(self._clients.values())

    def obtenir_client(self, id_client):
        return self._clients.get(id_client)

    def rechercher_clients(self, terme):
        terme = terme.lower().strip()
        if not terme:
            return self.lister_clients()
        return [
            c for c in self._clients.values()
            if terme in c.nom.lower() or terme in c.email.lower()
        ]

    def clients_sans_commande(self):
        return [c for c in self._clients.values() if not c.commandes]

    def clients_par_depense_decroissante(self):
        return sorted(self._clients.values(), key=lambda c: c.total_depense(), reverse=True)

    def meilleur_client(self):
        if not self._clients:
            return None
        return max(self._clients.values(), key=lambda c: c.total_depense())

    def modifier_client(self, id_client, nom=None, email=None, telephone=None):
        client = self._clients.get(id_client)
        if client is None:
            raise ValueError(f"Client inconnu (id={id_client}).")

        if email is not None and email != client.email:
            if not valider_email(email):
                raise ValueError(f"Email invalide : {email}")
            for autre in self._clients.values():
                if autre.id_client != id_client and autre.email == email:
                    raise ValueError(f"Un autre client utilise déjà l'email {email}.")
            client.email = email

        if telephone is not None:
            if not valider_telephone(telephone):
                raise ValueError(f"Téléphone invalide : {telephone}")
            client.telephone = telephone

        if nom is not None:
            if not nom.strip():
                raise ValueError("Le nom ne peut pas être vide.")
            client.nom = nom.strip()

        return client

    def supprimer_client(self, id_client):
        client = self._clients.get(id_client)
        if client is None:
            raise ValueError(f"Client inconnu (id={id_client}).")
        if client.commandes:
            raise ValueError(
                f"Impossible de supprimer '{client.nom}' : il a {len(client.commandes)} "
                "commande(s) enregistrée(s). Envisagez de l'archiver plutôt."
            )
        del self._clients[id_client]

    def exporter_clients_csv(self, chemin_fichier):
        with open(chemin_fichier, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["ID", "Nom", "Email", "Téléphone", "Total dépensé (€)"])
            for client in self.lister_clients():
                writer.writerow([client.id_client, client.nom, client.email,
                                  client.telephone, client.total_depense()])

Overwriting gestion_commerciale.py


In [4]:
%%writefile interface_gestion.py
"""
Interface graphique Tkinter — version finale de la semaine 1.
Lancer avec : python interface_gestion.py
"""
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from gestion_commerciale import GestionCommerciale


class ApplicationGestion:
    def __init__(self, fenetre):
        self.gestion = GestionCommerciale()
        self.fenetre = fenetre
        self.fenetre.title("Gestion Clients / Commandes")
        self.fenetre.geometry("700x550")

        self._construire_formulaire()
        self._construire_recherche()
        self._construire_tableau()
        self._construire_actions()

    def _construire_formulaire(self):
        cadre = tk.LabelFrame(self.fenetre, text="Ajouter un client", padx=10, pady=10)
        cadre.pack(padx=10, pady=5, fill="x")

        tk.Label(cadre, text="Nom :").grid(row=0, column=0, sticky="e", padx=5, pady=3)
        self.champ_nom = tk.Entry(cadre, width=25)
        self.champ_nom.grid(row=0, column=1, padx=5, pady=3)

        tk.Label(cadre, text="Email :").grid(row=1, column=0, sticky="e", padx=5, pady=3)
        self.champ_email = tk.Entry(cadre, width=25)
        self.champ_email.grid(row=1, column=1, padx=5, pady=3)

        tk.Label(cadre, text="Téléphone :").grid(row=2, column=0, sticky="e", padx=5, pady=3)
        self.champ_telephone = tk.Entry(cadre, width=25)
        self.champ_telephone.grid(row=2, column=1, padx=5, pady=3)

        tk.Button(cadre, text="Ajouter", command=self.ajouter_client).grid(
            row=3, column=0, columnspan=2, pady=8)

    def _construire_recherche(self):
        cadre = tk.Frame(self.fenetre, padx=10)
        cadre.pack(fill="x")
        tk.Label(cadre, text="Rechercher :").pack(side="left")
        self.champ_recherche = tk.Entry(cadre, width=30)
        self.champ_recherche.pack(side="left", padx=5)
        self.champ_recherche.bind("<KeyRelease>", lambda event: self._rafraichir_tableau())

    def _construire_tableau(self):
        cadre = tk.LabelFrame(self.fenetre, text="Liste des clients", padx=10, pady=10)
        cadre.pack(padx=10, pady=5, fill="both", expand=True)

        colonnes = ("id", "nom", "email", "telephone", "total_depense")
        self.tableau = ttk.Treeview(cadre, columns=colonnes, show="headings")
        libelles = {"id": "ID", "nom": "Nom", "email": "Email",
                    "telephone": "Téléphone", "total_depense": "Total dépensé (€)"}
        for col in colonnes:
            self.tableau.heading(col, text=libelles[col])
            self.tableau.column(col, width=120)
        self.tableau.pack(fill="both", expand=True)

    def _construire_actions(self):
        cadre = tk.Frame(self.fenetre, pady=10)
        cadre.pack(fill="x")
        tk.Button(cadre, text="Exporter en CSV", command=self.exporter_csv).pack(side="left", padx=10)
        tk.Button(cadre, text="Supprimer le client sélectionné", command=self.supprimer_client).pack(
            side="left", padx=10)

    def ajouter_client(self):
        nom = self.champ_nom.get()
        email = self.champ_email.get()
        telephone = self.champ_telephone.get()
        try:
            self.gestion.ajouter_client(nom, email, telephone)
        except ValueError as erreur:
            messagebox.showerror("Erreur de saisie", str(erreur))
            return
        self._rafraichir_tableau()
        self.champ_nom.delete(0, tk.END)
        self.champ_email.delete(0, tk.END)
        self.champ_telephone.delete(0, tk.END)
        messagebox.showinfo("Succès", "Client ajouté avec succès.")

    def supprimer_client(self):
        selection = self.tableau.selection()
        if not selection:
            messagebox.showwarning("Attention", "Sélectionnez un client dans le tableau.")
            return
        id_client = int(self.tableau.item(selection[0])["values"][0])
        try:
            self.gestion.supprimer_client(id_client)
        except ValueError as erreur:
            messagebox.showerror("Suppression impossible", str(erreur))
            return
        self._rafraichir_tableau()

    def exporter_csv(self):
        chemin = filedialog.asksaveasfilename(defaultextension=".csv", filetypes=[("Fichier CSV", "*.csv")])
        if not chemin:
            return
        self.gestion.exporter_clients_csv(chemin)
        messagebox.showinfo("Export réussi", f"Clients exportés vers :\n{chemin}")

    def _rafraichir_tableau(self):
        for ligne in self.tableau.get_children():
            self.tableau.delete(ligne)
        terme = self.champ_recherche.get()
        for client in self.gestion.rechercher_clients(terme):
            self.tableau.insert("", tk.END, values=(
                client.id_client, client.nom, client.email,
                client.telephone, client.total_depense()
            ))


if __name__ == "__main__":
    fenetre = tk.Tk()
    app = ApplicationGestion(fenetre)
    fenetre.mainloop()

Overwriting interface_gestion.py


### Testez dans un terminal
```bash
python interface_gestion.py
```
Testez la recherche en direct, la suppression (avec et sans commande), et l'export CSV.

---
## 5. Exercice 


### À faire :

1. Listez, dans une cellule Markdown ci-dessous, les **fonctionnalités actuellement développées** de votre application (reprenez votre cahier des charges du Jour 1 et cochez ce qui est fait)
2. Listez les **bugs ou limites connues** que vous avez identifiés en testant (soyez honnête — c'est une compétence professionnelle essentielle que d'identifier les failles de son propre travail)
3. Proposez **2 fonctionnalités** que vous ajouteriez si vous aviez plus de temps

EXERCICE : à compléter dans cette cellule (double-cliquez pour éditer)

**1. Fonctionnalités développées :**
- [ ] ...

**2. Bugs ou limites connues :**
- ...

**3. Propositions d'amélioration :**
- ...